# CNN for Stock Price Movement Prediction
## Optimized Version with BCEWithLogitsLoss and Proper Class Balancing

Based on D2L Principles - This notebook implements a CNN to predict binary stock price movements using chart images.

## Phase 1: Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import os.path as op
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)
from matplotlib import pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Image dimensions configuration
IMAGE_WIDTH = {5: 15, 20: 60, 60: 180}
IMAGE_HEIGHT = {5: 32, 20: 64, 60: 96}

# Configuration
IMG_PATH = "C:\\Users\\eblankso\\Downloads\\monthly_20d"
YEAR = 2017
WINDOW = 20  # 20-day window

# Hyperparameter configuration
CONFIG = {
    'batch_size': 32,
    'learning_rate': 0.001,  # Will be tuned by LR finder
    'dropout_rate': 0.5,
    'num_epochs': 50,
    'weight_decay': 1e-4,  # Increased for better regularization
    'grad_clip': 1.0,
}

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Phase 2: Data Loading

In [ ]:
print("\n" + "=" * 60)
print("PHASE 2: DATA LOADING")
print("=" * 60)

# Load images
images = np.memmap(
    op.join(IMG_PATH, f"20d_month_has_vb_[20]_ma_{YEAR}_images.dat"),
    dtype=np.uint8,
    mode='r'
).reshape((-1, IMAGE_HEIGHT[WINDOW], IMAGE_WIDTH[WINDOW]))

print(f"Images shape: {images.shape}")

# Load labels
label_df = pd.read_feather(
    op.join(IMG_PATH, f"20d_month_has_vb_[20]_ma_{YEAR}_labels_w_delay.feather")
)

print(f"Labels shape: {label_df.shape}")
print(f"Columns: {label_df.columns.tolist()}")

# Create binary target: 1 if price goes up, 0 if down
label_df['Ret_5d_binary'] = (label_df['Ret_5d'] > 0).astype(int)

print(f"\nClass distribution:")
print(label_df['Ret_5d_binary'].value_counts(normalize=True))

## Phase 3: Data Preprocessing and Splitting

In [ ]:
print("\n" + "=" * 60)
print("PHASE 3: DATA PREPROCESSING")
print("=" * 60)

# Handle missing values
missing_count = label_df['Ret_5d'].isnull().sum()
print(f"Missing Ret_5d values: {missing_count}")

if missing_count > 0:
    valid_mask = ~label_df['Ret_5d'].isnull()
    images = images[valid_mask.values]
    label_df = label_df[valid_mask].reset_index(drop=True)

n_samples = len(images)
print(f"Total valid samples: {n_samples:,}")

assert len(images) == len(label_df), "Images and labels count mismatch!"

## Phase 4: Time-Series Split (Chronological Order)

In [ ]:
print("\n" + "=" * 60)
print("PHASE 4: TIME-SERIES SPLIT")
print("=" * 60)

# Sort by date to maintain temporal order
sorted_indices = label_df.sort_values('Date').index.values
label_df = label_df.iloc[sorted_indices].reset_index(drop=True)
images = images[sorted_indices]

# Split by unique dates: 60% train, 20% validation, 20% test
unique_dates = label_df['Date'].unique()
n_dates = len(unique_dates)

train_cutoff = unique_dates[int(0.6 * n_dates) - 1]
val_cutoff = unique_dates[int(0.8 * n_dates) - 1]

train_mask = label_df['Date'] <= train_cutoff
val_mask = (label_df['Date'] > train_cutoff) & (label_df['Date'] <= val_cutoff)
test_mask = label_df['Date'] > val_cutoff

# Split data
train_images, train_labels = images[train_mask], label_df[train_mask]
val_images, val_labels = images[val_mask], label_df[val_mask]
test_images, test_labels = images[test_mask], label_df[test_mask]

print(f"Train: {len(train_images):,} samples ({len(train_images)/n_samples*100:.1f}%)")
print(f"Val:   {len(val_images):,} samples ({len(val_images)/n_samples*100:.1f}%)")
print(f"Test:  {len(test_images):,} samples ({len(test_images)/n_samples*100:.1f}%)")

# Verify no temporal leakage
assert train_labels['Date'].max() < val_labels['Date'].min()
assert val_labels['Date'].max() < test_labels['Date'].min()
print("\n✓ Time-series split verified: No temporal data leakage")

## Phase 5: Convert to PyTorch Tensors

In [ ]:
print("\n" + "=" * 60)
print("PHASE 5: TENSOR CONVERSION")
print("=" * 60)

# Convert to PyTorch tensors and normalize
X_train = torch.tensor(train_images, dtype=torch.float32).unsqueeze(1) / 255.0
y_train = torch.tensor(train_labels['Ret_5d_binary'].values, dtype=torch.float32)

X_val = torch.tensor(val_images, dtype=torch.float32).unsqueeze(1) / 255.0
y_val = torch.tensor(val_labels['Ret_5d_binary'].values, dtype=torch.float32)

X_test = torch.tensor(test_images, dtype=torch.float32).unsqueeze(1) / 255.0
y_test = torch.tensor(test_labels['Ret_5d_binary'].values, dtype=torch.float32)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Normalized range: [{X_train.min():.3f}, {X_train.max():.3f}]")

## Phase 6: Data Augmentation Dataset

In [ ]:
print("\n" + "=" * 60)
print("PHASE 6: DATA AUGMENTATION")
print("=" * 60)

class AugmentedDataset(Dataset):
    """Dataset with optional augmentation for training."""
    
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]
        
        if self.augment:
            # Horizontal flip with 30% probability (time reversal)
            if np.random.random() < 0.3:
                x = torch.flip(x, dims=[2])
            # Add small noise with 30% probability
            if np.random.random() < 0.3:
                noise = torch.randn_like(x) * 0.01
                x = torch.clamp(x + noise, 0, 1)
        
        return x, y

## Phase 7: Create DataLoaders

In [ ]:
print("\n" + "=" * 60)
print("PHASE 7: DATALOADERS")
print("=" * 60)

# Create augmented training dataset
train_dataset = AugmentedDataset(X_train, y_train, augment=True)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

## Phase 8: Model Architecture

In [ ]:
print("\n" + "=" * 60)
print("PHASE 8: MODEL ARCHITECTURE")
print("=" * 60)

class StockCNN(nn.Module):
    """CNN for stock price movement prediction with BCEWithLogitsLoss."""
    
    def __init__(self, dropout_rate=0.5, input_height=64, input_width=60):
        super(StockCNN, self).__init__()
        
        # Convolutional layers with batch normalization
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout_rate)
        
        # Calculate flattened size
        h = input_height // 4  # Two pooling layers
        w = input_width // 4
        self.flat_size = 32 * h * w
        
        # Fully connected layers (no sigmoid at end!)
        self.fc1 = nn.Linear(self.flat_size, 64)
        self.bn_fc = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 1)  # Raw logits output
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        
        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully connected (no sigmoid - raw logits for BCEWithLogitsLoss)
        x = self.dropout(F.relu(self.bn_fc(self.fc1(x))))
        x = self.fc2(x)
        
        return x

# Create model
model = StockCNN(dropout_rate=CONFIG['dropout_rate'])

# Verify model
test_input = torch.randn(2, 1, 64, 60)
test_output = model(test_input)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape} (raw logits)")
print(f"Output range: [{test_output.min().item():.4f}, {test_output.max().item():.4f}]")

# Move model to device
model = model.to(device)
print(f"\n✓ Model moved to {device}")

## Phase 9: Loss Function with Class Weights

In [ ]:
print("\n" + "=" * 60)
print("PHASE 9: LOSS FUNCTION")
print("=" * 60)

# Calculate pos_weight for BCEWithLogitsLoss (handles class imbalance)
y_train_np = y_train.numpy()
n_pos = (y_train_np == 1).sum()
n_neg = (y_train_np == 0).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

print(f"Class 0 (DOWN): {n_neg} samples")
print(f"Class 1 (UP):   {n_pos} samples")
print(f"pos_weight: {pos_weight.item():.4f}")

# Use BCEWithLogitsLoss (more stable than BCE + sigmoid)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer with weight decay for regularization
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

print(f"\n✓ Loss function: BCEWithLogitsLoss with pos_weight={pos_weight.item():.4f}")
print(f"✓ Optimizer: Adam (lr={CONFIG['learning_rate']}, weight_decay={CONFIG['weight_decay']})")

## Phase 10: Training Functions

In [ ]:
print("\n" + "=" * 60)
print("PHASE 10: TRAINING FUNCTIONS")
print("=" * 60)

def train_epoch(model, loader, criterion, optimizer, device, max_grad_norm=1.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        # Move to device
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass - raw logits
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        
        # Backward pass with gradient clipping
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        
        # Statistics (apply sigmoid for accuracy calculation)
        total_loss += loss.item()
        probs = torch.sigmoid(outputs)
        predicted = (probs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), 100 * correct / total, np.array(all_probs), np.array(all_labels)


def find_optimal_threshold(val_probs, val_labels):
    """Find threshold that maximizes F1 score."""
    thresholds = np.arange(0.3, 0.7, 0.02)
    f1_scores = []
    
    for thresh in thresholds:
        preds = (val_probs > thresh).astype(int)
        f1_scores.append(f1_score(val_labels, preds, zero_division=0))
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    return best_threshold, best_f1, thresholds, f1_scores

print("✓ Training functions defined")

## Phase 11: Learning Rate Finder (Optional)

In [ ]:
print("\n" + "=" * 60)
print("PHASE 11: LEARNING RATE FINDER (Quick Test)")
print("=" * 60)

def find_lr(model, loader, criterion, device, init_lr=1e-5, final_lr=1.0, num_iter=100):
    """Find optimal learning rate."""
    # Create a copy of model parameters
    model_copy = StockCNN(dropout_rate=CONFIG['dropout_rate']).to(device)
    model_copy.load_state_dict(model.state_dict())
    
    optimizer = optim.Adam(model_copy.parameters(), lr=init_lr)
    lrs = []
    losses = []
    
    mult = (final_lr / init_lr) ** (1 / num_iter)
    
    model_copy.train()
    for i, (inputs, labels) in enumerate(loader):
        if i >= num_iter:
            break
        
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_copy(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        lrs.append(optimizer.param_groups[0]['lr'])
        losses.append(loss.item())
        
        for param_group in optimizer.param_groups:
            param_group['lr'] *= mult
    
    return lrs, losses

# Quick LR test (optional - comment out if you want to skip)
try:
    print("Running quick LR test (20 iterations)...")
    lrs, losses = find_lr(model, train_loader, criterion, device, num_iter=20)
    
    # Find recommended LR (steepest gradient)
    if len(losses) > 10:
        import numpy as np
        log_lrs = np.log(lrs)
        gradients = np.gradient(losses, log_lrs)
        steepest_idx = np.argmin(gradients[5:-5]) + 5
        recommended_lr = lrs[steepest_idx]
        print(f"✓ Recommended learning rate: {recommended_lr:.6f}")
        print(f"  Current LR: {CONFIG['learning_rate']}")
        
        # Optionally update config (uncomment to use)
        # CONFIG['learning_rate'] = recommended_lr
        # print(f"  Updated LR to: {CONFIG['learning_rate']}")
except Exception as e:
    print(f"LR finder skipped: {e}")

## Phase 12: Training Loop

In [ ]:
print("\n" + "=" * 60)
print("PHASE 12: TRAINING")
print("=" * 60)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}
best_val_acc = 0.0
best_model_state = None
patience = 10
patience_counter = 0

print(f"Training for {CONFIG['num_epochs']} epochs...")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Gradient clipping: {CONFIG['grad_clip']}")
print("-" * 60)

for epoch in range(CONFIG['num_epochs']):
    # Training
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, CONFIG['grad_clip']
    )
    
    # Validation
    val_loss, val_acc, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device
    )
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Check if best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1
        marker = ""
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{CONFIG['num_epochs']}: "
              f"Train Loss={train_loss:.4f}, Acc={train_acc:.2f}% | "
              f"Val Loss={val_loss:.4f}, Acc={val_acc:.2f}%{marker}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("-" * 60)
print(f"Best validation accuracy: {best_val_acc:.2f}%")

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    torch.save(best_model_state, 'best_model.pth')
    print("✓ Best model saved")

## Phase 13: Threshold Optimization

In [ ]:
print("\n" + "=" * 60)
print("PHASE 13: THRESHOLD OPTIMIZATION")
print("=" * 60)

# Get final validation predictions
model.eval()
val_probs, val_labels = [], []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze()
        probs = torch.sigmoid(outputs)
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(labels.numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

# Find optimal threshold
best_threshold, best_f1, thresholds, f1_scores = find_optimal_threshold(val_probs, val_labels)

print(f"Default threshold (0.50): F1={f1_scores[np.abs(thresholds-0.5).argmin()]:.4f}")
print(f"Optimal threshold ({best_threshold:.2f}): F1={best_f1:.4f}")

# Plot threshold optimization
plt.figure(figsize=(8, 5))
plt.plot(thresholds, f1_scores, 'b-o', linewidth=2, markersize=8)
plt.axvline(x=best_threshold, color='green', linestyle='--', 
            label=f'Optimal ({best_threshold:.2f})')
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('Threshold Optimization for F1 Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 14: Test Set Evaluation

In [ ]:
print("\n" + "=" * 60)
print("PHASE 14: TEST SET EVALUATION")
print("=" * 60)

# Evaluate on test set with optimal threshold
model.eval()
test_probs = []
test_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze()
        probs = torch.sigmoid(outputs)
        test_probs.extend(probs.cpu().numpy())
        test_labels.extend(labels.numpy())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds = (test_probs > best_threshold).astype(float)

# Calculate metrics
accuracy = accuracy_score(test_labels, test_preds)
precision = precision_score(test_labels, test_preds, zero_division=0)
recall = recall_score(test_labels, test_preds, zero_division=0)
f1 = f1_score(test_labels, test_preds, zero_division=0)

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {accuracy*100:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Down', 'Up'], digits=4))

## Phase 15: Confusion Matrix

In [ ]:
print("\n" + "=" * 60)
print("PHASE 15: CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Down', 'Up'],
    yticklabels=['Down', 'Up'],
    annot_kws={'size': 14}
)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives (TN):  {tn:,} (correctly predicted DOWN)")
print(f"  False Positives (FP): {fp:,} (predicted UP, was DOWN)")
print(f"  False Negatives (FN): {fn:,} (predicted DOWN, was UP)")
print(f"  True Positives (TP):  {tp:,} (correctly predicted UP)")

## Phase 16: Training Curves

In [ ]:
print("\n" + "=" * 60)
print("PHASE 16: TRAINING CURVES")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

# Loss curves
ax1.plot(epochs, history['train_loss'], 'b-', label='Training', linewidth=2)
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(epochs, history['train_acc'], 'b-', label='Training', linewidth=2)
ax2.plot(epochs, history['val_acc'], 'r-', label='Validation', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Overfitting analysis
final_train_acc = history['train_acc'][-1]
final_val_acc = history['val_acc'][-1]
gap = final_train_acc - final_val_acc

print(f"\nOverfitting Analysis:")
print(f"  Final Train Accuracy: {final_train_acc:.2f}%")
print(f"  Final Val Accuracy:   {final_val_acc:.2f}%")
print(f"  Gap:                  {gap:.2f}%")

if gap > 10:
    print("  ⚠️  OVERFITTING DETECTED - Consider increasing dropout or weight decay")
elif gap < 0:
    print("  ⚠️  UNDERFITTING DETECTED - Model too simple")
else:
    print("  ✓ Model generalizing reasonably well")

## Phase 17: Misclassification Analysis

In [ ]:
print("\n" + "=" * 60)
print("PHASE 17: MISCLASSIFICATION ANALYSIS")
print("=" * 60)

misclassified_mask = test_preds != test_labels
misclassified_idx = np.where(misclassified_mask)[0]
false_positives = np.where((test_preds == 1) & (test_labels == 0))[0]
false_negatives = np.where((test_preds == 0) & (test_labels == 1))[0]

print(f"Total test samples: {len(test_labels):,}")
print(f"Misclassified: {len(misclassified_idx):,} ({len(misclassified_idx)/len(test_labels)*100:.2f}%)")
print(f"  False Positives: {len(false_positives):,}")
print(f"  False Negatives: {len(false_negatives):,}")

# Confidence analysis
misclassified_conf = np.abs(test_probs[misclassified_mask] - 0.5) * 2
correct_conf = np.abs(test_probs[~misclassified_mask] - 0.5) * 2

print(f"\nConfidence Analysis:")
print(f"  Avg confidence on CORRECT predictions: {correct_conf.mean():.4f}")
print(f"  Avg confidence on ERRORS:              {misclassified_conf.mean():.4f}")

if misclassified_conf.mean() > 0.5:
    print("  ⚠️  Model is overconfident on errors!")
else:
    print("  ✓ Model is less confident on errors (good)")

# Visualize misclassified examples
if len(misclassified_idx) > 0:
    n_samples = min(9, len(misclassified_idx))
    sample_idx = np.random.choice(misclassified_idx, n_samples, replace=False)
    
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    axes = axes.ravel()
    
    for i, idx in enumerate(sample_idx):
        img = X_test[idx].squeeze().numpy()
        true_label = 'UP' if test_labels[idx] == 1 else 'DOWN'
        pred_label = 'UP' if test_preds[idx] == 1 else 'DOWN'
        conf = abs(test_probs[idx] - 0.5) * 2
        
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"True: {true_label}\nPred: {pred_label}\nConf: {conf:.2f}", 
                         color='red', fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('misclassified_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

## Phase 18: Summary and Recommendations

In [ ]:
print("\n" + "=" * 60)
print("PHASE 18: SUMMARY AND RECOMMENDATIONS")
print("=" * 60)

print(f"""
FINAL PERFORMANCE:
- Test Accuracy:  {accuracy*100:.2f}%
- F1-Score:      {f1:.4f}
- Precision:     {precision:.4f}
- Recall:        {recall:.4f}
- Optimal Threshold: {best_threshold:.2f}
""")

print("RECOMMENDATIONS FOR IMPROVEMENT:")

if accuracy < 0.55:
    print("  1. INCREASE MODEL CAPACITY: Add more convolutional layers or increase filters")
if gap > 10:
    print("  2. REDUCE OVERFITTING: Increase dropout (0.6-0.7) or weight decay (5e-4)")
if abs(precision - recall) > 0.15:
    print("  3. BALANCE PRECISION/RECALL: Adjust threshold or use focal loss")
if len(false_negatives) > len(false_positives) * 1.5:
    print("  4. IMPROVE UP DETECTION: Increase pos_weight or add more UP samples")
elif len(false_positives) > len(false_negatives) * 1.5:
    print("  4. REDUCE FALSE POSITIVES: Lower pos_weight or adjust threshold")

print("""
ADDITIONAL SUGGESTIONS:
  5. DATA AUGMENTATION: Add rotation, scaling, or more aggressive transformations
  6. ENSEMBLE METHODS: Train multiple models with different seeds
  7. TRANSFER LEARNING: Use pre-trained ResNet18 or EfficientNet
  8. FEATURE ENGINEERING: Combine images with technical indicators
""")

print("\n" + "=" * 60)
print("✓ ANALYSIS COMPLETE")
print("=" * 60)